# 05 - The three physical flexibility limits and the active limiter (workflow steps 5-6)

**Question:** are the energy/health limit $\mathcal F^{en}_k$, the degradation limit
$\mathcal F^{deg}_k$ and the network-deliverability limit $\mathcal F^{net}_k$ each computed
correctly, and is the physical envelope verifiably their intersection, with the active limiter
identified correctly -- *before* any EMS uses them?

1. $\mathcal F^{net}_k$ for the nominal forecast (PyDAE bisection) -- cached for all later notebooks.
2. Independent re-verification of every $P^{net}_k$ (feasible at $P^{net}$, infeasible just above).
3. $P^{en}$ against its closed form; $P^{deg}$ feasibility and maximality.
4. Brute-force check that $[-P^{ch}_{flex},P^{dis}_{flex}] = \mathcal F^{en}\cap\mathcal F^{deg}\cap\mathcal F^{net}$
   on a fine power grid (network membership evaluated with PyDAE), and that the limiter is the argmin.

Outputs: `output/results/precompute_network_forecast.csv`, `output/results/results_envelope_validation.csv`,
`output/results/results_intersection_bruteforce.csv`, `output/experiment_figures/Fig_flexibility_limits.pdf`.

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dfm_acdc.config import RESULTS_DIR, FIG_DIR
from dfm_acdc.simulate import Study
from dfm_acdc.envelope import (energy_flex, degradation_flex, degradation_allowance, build_envelope,
                               _deg_cost)
from dfm_acdc import plotstyle

plotstyle.apply_style()
S = Study()
b, dfm, deg = S.sp.b, S.sp.dfm, S.deg
PRE_PATH = os.path.join(RESULTS_DIR, "precompute_network_forecast.csv")
tests = []
def check(name, value, ok, tol=""):
    tests.append({"test": name, "value": value, "tolerance": tol, "pass": bool(ok)})
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {value}")

## 1. Network-deliverable flexibility for the nominal forecast

In [ ]:
t0 = time.perf_counter()
pre = S.precompute(verbose=True)
pre.to_csv(PRE_PATH, index=False)
print(f"precomputation: {time.perf_counter()-t0:.1f} s, {S.net.n_solves} PyDAE solves")
print(pre[["t_h", "P_net_dis", "net_bind_dis", "P_net_ch", "net_bind_ch", "C0"]].iloc[::8].round(1).to_string(index=False))

## 2. Independent re-verification of every $P^{net}_k$

In [ ]:
orc = S.oracle
bad = 0
for k in range(96):
    a0 = pre.alpha0[k]
    base = np.maximum(orc.evaluate(k, 0.0, a0)["e"], 0.0)
    for side, col in ((+1, "P_net_dis"), (-1, "P_net_ch")):
        P = pre[col][k]
        ok_at = np.all(orc.evaluate(k, side * P, a0)["e"] <= base + 1e-9)
        ok_above = True if P >= b.P_rated_kw - 1e-9 else not np.all(orc.evaluate(k, side * (P + 1.0), a0)["e"] <= base + 1e-9)
        bad += (not ok_at) + (not ok_above)
check("P_net feasible at P_net and infeasible at P_net + 1 kW (192 limits)", f"{bad} failures", bad == 0)

## 3. Energy and degradation limits

In [ ]:
E_use = b.soh_0 * b.E_rated_kwh
F0 = deg.F_from_soh(b.soh_0)
err = 0.0
for soc in np.linspace(b.soc_min, b.soc_max, 81):
    d, c = energy_flex(soc, E_use, b, dfm.H_f_h)
    err = max(err, abs(d - min(b.P_rated_kw, (soc - b.soc_min) * E_use * b.eta_dis / dfm.H_f_h)),
              abs(c - min(b.P_rated_kw, (b.soc_max - soc) * E_use / (b.eta_ch * dfm.H_f_h))))
check("P_en matches Eqs. energy_dis_flex / energy_ch_flex", err, err < 1e-12)

D_full, D_max, _ = degradation_allowance(deg, F0, 0.0, dfm)
viol_feas = viol_max = 0
for soc in np.linspace(0.15, 0.85, 15):
    for frac in (1.0, 0.5, 0.2, 0.05, 0.0):
        Da = frac * D_full
        pd_, pc_ = degradation_flex(deg, soc, E_use, b, dfm.H_f_h, F0, Da)
        for side, P in ((+1, pd_), (-1, pc_)):
            if _deg_cost(deg, soc, E_use, b, dfm.H_f_h, F0, P, side) > Da + 1e-15:
                viol_feas += 1
            if P < b.P_rated_kw - 1e-6 and _deg_cost(deg, soc, E_use, b, dfm.H_f_h, F0, P + 0.01, side) <= Da:
                viol_max += 1
check("P_deg feasible: dD(P_deg sustained over H_f) <= D_allow", f"{viol_feas} failures", viol_feas == 0)
check("P_deg maximal: dD(P_deg + 0.01 kW) > D_allow", f"{viol_max} failures", viol_max == 0)
print(f"daily cycle-degradation allowance at SOH_0={b.soh_0}: D_max = {D_max:.3e} (SOH units)")

## 4. Brute force: physical envelope = intersection of the three sets

In [ ]:
grid = np.linspace(-b.P_rated_kw, b.P_rated_kw, 121)          # 2.5-kW resolution
cases = [(k, soc, frac) for k in (4, 32, 46, 60, 74, 80, 90) for soc, frac in ((0.25, 1.0), (0.75, 0.15))]
rows, mism, lim_bad = [], 0, 0
for k, soc, frac in cases:
    Da = frac * D_full
    F_cyc_day = (D_max - Da) / float(deg.dL(F0))
    env = build_envelope(b, deg, dfm, soc, E_use, F0, F_cyc_day, pre.P_net_dis[k], pre.P_net_ch[k], 0.0, 0.0)
    a0 = pre.alpha0[k]
    base = np.maximum(orc.evaluate(k, 0.0, a0)["e"], 0.0)
    member = []
    for P in grid:
        in_en = (-env.P_en_ch - 1e-9 <= P <= env.P_en_dis + 1e-9)
        side = 1 if P >= 0 else -1
        in_deg = _deg_cost(deg, soc, E_use, b, dfm.H_f_h, F0, abs(P), side) <= Da + 1e-15
        in_net = bool(np.all(orc.evaluate(k, P, a0)["e"] <= base + 1e-9))
        member.append(in_en and in_deg and in_net)
    member = np.array(member)
    in_box = (grid >= -env.P_flex_ch - 1e-9) & (grid <= env.P_flex_dis + 1e-9)
    n_mis = int(np.sum(member != in_box))
    mism += n_mis
    for side in (+1, -1):
        vals = {"en": env.P_en_dis if side > 0 else env.P_en_ch, "deg": env.P_deg_dis if side > 0 else env.P_deg_ch,
                "net": env.P_net_dis if side > 0 else env.P_net_ch}
        exp_lim = "rated" if min(vals.values()) >= b.P_rated_kw - 1e-6 else min(vals, key=vals.get)
        lim_bad += env.limiter(side) != exp_lim
    rows.append({"k": k, "t_h": k * 0.25, "soc": soc, "D_allow_frac": frac, **env.as_dict(), "grid_mismatches": n_mis})
bf = pd.DataFrame(rows)
bf.to_csv(os.path.join(RESULTS_DIR, "results_intersection_bruteforce.csv"), index=False)
print(bf[["t_h", "soc", "D_allow_frac", "P_en_dis", "P_deg_dis", "P_net_dis", "P_flex_dis", "lim_dis",
          "P_en_ch", "P_deg_ch", "P_net_ch", "P_flex_ch", "lim_ch", "grid_mismatches"]].round(1).to_string(index=False))
check(f"intersection == [-P_flex_ch, P_flex_dis] on a 2.5-kW grid ({len(cases)} states x 121 points)",
      f"{mism} mismatching points", mism == 0)
check("active limiter == argmin of the three limits", f"{lim_bad} errors", lim_bad == 0)
print("limiters seen in the brute-force set:", sorted(set(bf.lim_dis) | set(bf.lim_ch)))

In [ ]:
val = pd.DataFrame(tests)
val.to_csv(os.path.join(RESULTS_DIR, "results_envelope_validation.csv"), index=False)
print(f"\n{val['pass'].sum()}/{len(val)} tests passed")
assert val["pass"].all()

Because each set is an interval containing 0 (energy and degradation costs are monotone in
$|P|$; the network check is monotone over the tested range), the intersection is itself an
interval and equals the min of the individual bounds -- verified point by point above.

## Supplementary figure -- network deliverability and battery-side limits

In [ ]:
FIG_WIDTH, FIG_HEIGHT = plotstyle.IEEE_PAGE_WIDTH_IN, 2.6
fig, axs = plt.subplots(1, 2, figsize=(FIG_WIDTH, FIG_HEIGHT), gridspec_kw={"width_ratios": [1.4, 1.0]})
ax = axs[0]
t = pre.t_h.values
ax.step(t, pre.P_net_dis, where="post", color=plotstyle.CAT["violet"], label=r"$P^{dis}_{net,k}$")
ax.step(t, -pre.P_net_ch, where="post", color=plotstyle.CAT["violet"], ls="--", label=r"$-P^{ch}_{net,k}$")
ax.axhline(b.P_rated_kw, color=plotstyle.INK_MUTED, lw=0.8, ls=":"); ax.axhline(-b.P_rated_kw, color=plotstyle.INK_MUTED, lw=0.8, ls=":")
ax.fill_between(t, 0, pre.C0, step="post", color=plotstyle.STATUS["critical"], alpha=0.3, lw=0, label="required curtailment (B0)")
ax.set_xlim(0, 24); ax.set_xticks(range(0, 25, 4)); ax.set_xlabel("Time [h]"); ax.set_ylabel("Power [kW]")
for tt, g in zip(t, pre.net_bind_dis):
    if g == "S_vsc":
        ax.plot(tt + 0.125, b.P_rated_kw + 8, "|", color=plotstyle.CAT["violet"], ms=3)
for tt, g in zip(t, pre.net_bind_ch):
    if g == "I_feeder":
        ax.plot(tt + 0.125, -b.P_rated_kw - 8, "|", color=plotstyle.CAT["orange"], ms=3)
ax.text(0.3, b.P_rated_kw + 14, "binding: I01-H01 converter rating", fontsize=6, color=plotstyle.CAT["violet"])
ax.text(0.3, -b.P_rated_kw - 24, "binding: feeder-head current", fontsize=6, color=plotstyle.CAT["orange"])
ax.set_ylim(-b.P_rated_kw - 35, b.P_rated_kw + 30)
ax.legend(frameon=False, fontsize=6.3, loc="center left")
ax = axs[1]
socs = np.linspace(b.soc_min, b.soc_max, 161)
Pen = np.array([energy_flex(s, E_use, b, dfm.H_f_h)[0] for s in socs])
ax.plot(socs, Pen, color=plotstyle.CAT["blue"], label=r"$P^{dis}_{en}$")
for frac, ls in ((0.35, "-"), (0.2, "--"), (0.05, ":")):
    Pdg = np.array([degradation_flex(deg, s, E_use, b, dfm.H_f_h, F0, frac * D_full)[0] for s in socs])
    Pdg = np.where(Pdg < Pen, Pdg, np.nan)      # shown only where degradation is the binding battery-side limit
    ax.plot(socs, Pdg, color=plotstyle.CAT["orange"], ls=ls, label=fr"$P^{{dis}}_{{deg}}$, $D^{{allow}}={frac:g}D^{{max}}$")
ax.set_xlabel("SOC"); ax.set_ylabel("Discharge limit [kW]"); ax.set_ylim(0, b.P_rated_kw * 1.05)
ax.legend(frameon=False, fontsize=6.0, loc="upper left")
ax.text(0.98, 0.03, r"$P_{deg}$ drawn where $P_{deg}<P_{en}$", transform=ax.transAxes, ha="right", fontsize=6, color=plotstyle.INK_SECONDARY)
for a, lab in zip(axs, "ab"):
    a.text(-0.16 if lab == "a" else -0.22, 1.04, f"({lab})", transform=a.transAxes, fontsize=8, va="top")
plotstyle.savefig(fig, os.path.join(FIG_DIR, "Fig_flexibility_limits.pdf"))
plt.show()

## Takeaway

* $\mathcal F^{net}$: discharge deliverability is bounded by the **I01-H01 Vdc-forming converter
  rating** at night and over the PV hours (it collapses to ~0 kW at midday, when the converter
  already exports the DC surplus), while charge deliverability is bounded by the **feeder-head
  currents** at the evening peak. Every limit was re-verified independently.
* $\mathcal F^{en}$ follows its closed form; $\mathcal F^{deg}$ is feasible and maximal and
  shrinks as the allowance is consumed and as SOC moves deeper (convex cycle-depth stress).
* The physical envelope equals the intersection of the three sets point by point, and the
  active limiter is their argmin. The envelope is therefore validated independently of the EMS.